# Phase 5 & 6: Machine Learning + Model Evaluation
Predictive Maintenance of Industrial Equipment using IoT Sensor Data

**Task:** binary classification -- predict `Failure` (0/1) from sensor and machine-usage features.

**Models compared:** Logistic Regression (baseline), Decision Tree (interpretable), Random Forest (recommended primary model).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve)

df = pd.read_csv('../data/features_final.csv', parse_dates=['Timestamp'])
df.shape

## 5.1 Select features & target
Use only features available from the API's single sensor profile. History-derived rolling/deviation features and all target-derived features are excluded from the deployed model.


In [ ]:
drop_cols = ['Machine_ID','Timestamp','Failure']
# Explicit allowlist: features the stateless prediction API can calculate consistently.
feature_cols = [
    'Temperature','Pressure','Vibration','RPM','Current','Voltage',
    'Operating_Hours','Maintenance_Count','Machine_Age','Maintenance_Frequency',
    'OpHours_Band_1000-2000','OpHours_Band_2000-3000','OpHours_Band_3000-4000','OpHours_Band_4000+',
    'Age_Group_Mid(3-6y)','Age_Group_Old(6-9y)','Age_Group_VeryOld(9y+)'
]
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected model features: {missing}')
X = df[feature_cols].astype(float)
y = df['Failure'].astype(int)
print('Features used:', len(feature_cols))
print('Class balance:\n', y.value_counts(normalize=True))


## 5.2 Train/Test split by MACHINE (avoids data leakage)
As in the earlier project, splitting by row would let the same machine's readings appear in both train and test. Instead, entire machines are held out for testing.

In [ ]:
machines = df['Machine_ID'].unique()
np.random.seed(42)
np.random.shuffle(machines)
n_train = int(len(machines)*0.75)
train_machines, test_machines = machines[:n_train], machines[n_train:]

train_idx = df['Machine_ID'].isin(train_machines)
test_idx = df['Machine_ID'].isin(test_machines)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f'Train: {len(X_train)} rows from {len(train_machines)} machines')
print(f'Test:  {len(X_test)} rows from {len(test_machines)} machines')
print('Train failure rate:', y_train.mean().round(4), '| Test failure rate:', y_test.mean().round(4))

## 5.3 Scale features (needed for Logistic Regression)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5.4 Train three models
`class_weight='balanced'` is used throughout since Failure is a rare event (~0.9% of rows) -- without this, a model could get 99% accuracy by predicting 'no failure' every time, which is useless.

In [ ]:
models = {}

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
models['Logistic Regression'] = (log_reg, X_test_scaled)

dtree = DecisionTreeClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=10, random_state=42)
dtree.fit(X_train, y_train)
models['Decision Tree'] = (dtree, X_test)

rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = (rf, X_test)

print('All 3 models trained.')

## 5.5 Compare models on the held-out test machines

In [ ]:
results = []
for name, (model, X_te) in models.items():
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:,1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, proba)
    })
results_df = pd.DataFrame(results).round(3)
results_df

## 5.6 Select Random Forest as the primary model
Selection should be based on the newly calculated held-out metrics, especially recall and precision under class imbalance. Do not assume a model is best until results are regenerated after feature changes.


In [ ]:
best_model, X_te = models['Random Forest']
pred = best_model.predict(X_te)
proba = best_model.predict_proba(X_te)[:,1]

print(classification_report(y_test, pred, digits=3))

## 5.7 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Healthy','Predicted Failure'],
            yticklabels=['Actual Healthy','Actual Failure'])
plt.title('Confusion Matrix - Random Forest')
plt.tight_layout(); plt.savefig('../outputs/confusion_matrix.png', dpi=110); plt.show()

**Business interpretation:**
- **False Negatives** (actual failure predicted healthy) are the costliest mistake -- a machine that actually fails goes uninspected, risking unplanned downtime.
- **False Positives** (actual healthy predicted failure) cost only an unnecessary inspection -- inconvenient but far cheaper than a missed failure.
- This is why Recall is prioritized as the key metric for this business problem, even if it means accepting more false alarms.

## 5.8 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {roc_auc_score(y_test, proba):.3f})', color='#2980b9')
plt.plot([0,1],[0,1],'--',color='gray', label='Random guess')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend(); plt.tight_layout(); plt.savefig('../outputs/roc_curve.png', dpi=110); plt.show()

## 5.9 Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(9,6))
importances.head(15).sort_values().plot(kind='barh', color='#27ae60')
plt.title('Top 15 Feature Importances - Random Forest')
plt.tight_layout(); plt.savefig('../outputs/feature_importance.png', dpi=110); plt.show()
importances.head(10)

**Observation:** Review the newly generated feature-importance chart after training. Since history-derived and target-derived features are excluded from the deployed model, importances reflect only features available to the single-reading API.


## 5.10 Save predictions and metrics for the Power BI dashboard

In [ ]:
test_df = df[test_idx].copy()
test_df['Predicted_Failure'] = pred
test_df['Failure_Probability'] = proba
test_df.to_csv('../outputs/test_predictions.csv', index=False)

results_df.to_csv('../outputs/model_comparison.csv', index=False)
importances.reset_index().rename(columns={'index':'Feature',0:'Importance'}).to_csv('../outputs/feature_importance.csv', index=False)

import joblib
joblib.dump(best_model, '../outputs/random_forest_model.pkl')
joblib.dump(scaler, '../outputs/scaler.pkl')

print('Saved refreshed test_predictions.csv, model_comparison.csv, feature_importance.csv, random_forest_model.pkl and scaler.pkl')
